In [2]:
import cv2
import numpy as np

# Load grayscale images
img1 = cv2.imread('Datasets/dataset_flowerpot-master/full_dataset/P81019-151014.jpg', cv2.IMREAD_GRAYSCALE)  # Query image
img2 = cv2.imread('Datasets/dataset_flowerpot-master/full_dataset/P81019-151024.jpg', cv2.IMREAD_GRAYSCALE)  # Train image

# Resize (optional)
scale_percent = 10
resize = lambda img: cv2.resize(img, (int(img.shape[1]*scale_percent/100), int(img.shape[0]*scale_percent/100)))
img1 = resize(img1)
img2 = resize(img2)

# SIFT detector
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# BFMatcher with Lowe's ratio test
bf = cv2.BFMatcher()
matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]

# Proceed if enough good matches
if len(good_matches) > 10:
    # Extract matched keypoints
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Estimate homography
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    # Get the corners of img1
    h, w = img1.shape
    corners = np.float32([[0,0], [w,0], [w,h], [0,h]]).reshape(-1,1,2)
    projected_corners = cv2.perspectiveTransform(corners, M)

    # Convert to color image to draw in green
    img2_color = cv2.cvtColor(img2, cv2.COLOR_GRAY2BGR)
    cv2.polylines(img2_color, [np.int32(projected_corners)], isClosed=True, color=(0,255,0), thickness=3)

    # Draw matches (only inliers)
    matches_mask = mask.ravel().tolist()
    matched_img = cv2.drawMatches(img1, kp1, img2_color, kp2, good_matches, None,
                                  matchesMask=matches_mask, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
else:
    print("Not enough matches found.")
    matched_img = img2

# Show result
cv2.imshow('SIFT Homography', matched_img)
cv2.waitKey(0)
cv2.destroyAllWindows()
